In [ ]:
import pdfplumber
import re
import json 
import os
from pathlib import Path

In [ ]:
def extraer_filas_desde_texto(path_pdf: str) -> list:
    """
    Abre el PDF, extrae líneas de autor y sus proyectos, y devuelve una lista de
    diccionarios con 'nombre' y 'proyectos'.
    """
    filas = []
    # Patrones de texto a ignorar (cabeceras y líneas irrelevantes)
    ignore_patterns = [
        r"SECRETARÍA GENERAL",
        r"REGISTRO Y TRÁMITE DE PROYECTOS",
        r"DE LEY Y ACTOS LEGISLATIVOS",
        r"LEGISLATURA\s*\d{4}\s*-\s*\d{4}",
        r"PROYECTOS DE LEY POR AUTOR",
        r"NOMBRE\s+NUMERO.*",
        r"\(.*\d{4}.*\)"  # Fechas entre paréntesis
    ]
    ignore_re = re.compile("|".join(ignore_patterns), re.IGNORECASE)

    # Patrón para capturar opcional 'ACU', número y año
    patron = re.compile(r'(ACU\s*)?(\d{1,3})/(\d{4})C', re.IGNORECASE)

    # Leer el PDF línea a línea
    with pdfplumber.open(path_pdf) as pdf:
        lines = []
        for page in pdf.pages:
            text = page.extract_text() or ""
            for l in text.splitlines():
                lines.append(l.strip())

    current_author = None 
    numeros_texto = ""
 
    for line in lines:
        if not line:
            continue
        # Saltar cabeceras o líneas irrelevantes
        if ignore_re.search(line):
            continue

        # Detectar nombre de autor: línea completamente en mayúsculas sin dígitos
        if re.match(r'^[A-ZÁÉÍÓÚÑÜ ]+$', line) and not re.search(r'\d', line):
            # Si ya había un autor anterior, procesar su acumulado
            if current_author and numeros_texto:
                proyectos = []
                for m in patron.finditer(numeros_texto):
                    acu_flag = bool(m.group(1))
                    numero = int(m.group(2))
                    anio = int(m.group(3))
                    entry = {"numeroCamara": numero, "anioCamara": anio}
                    if acu_flag:
                        entry["acu"] = True
                    proyectos.append(entry)
                filas.append({"nombre": current_author, "proyectos": proyectos})
            # Iniciar nuevo autor
            current_author = line
            numeros_texto = ""
        else:
            # Acumular texto numérico para el autor actual
            numeros_texto += " " + line

    # Procesar el último autor al finalizar
    if current_author and numeros_texto:
        proyectos = []
        for m in patron.finditer(numeros_texto):
            acu_flag = bool(m.group(1))
            numero = int(m.group(2))
            anio = int(m.group(3))
            entry = {"numeroCamara": numero, "anioCamara": anio}
            if acu_flag:
                entry["acu"] = True
            proyectos.append(entry)
        filas.append({"nombre": current_author, "proyectos": proyectos})

    return filas


In [11]:

def procesar_pdf_texto_individual(path_pdf: str, carpeta_salida: str) -> None:
    """
    Procesa un PDF de entrada, extrae las filas (autores + proyectos),
    y genera un archivo JSON por cada fila en la carpeta de salida.
    """
    os.makedirs(carpeta_salida, exist_ok=True)
    base = Path(path_pdf).stem

    filas = extraer_filas_desde_texto(path_pdf)
    print(f"✅ Total de filas extraídas: {len(filas)}")

    for idx, fila in enumerate(filas, start=1):
        out = Path(carpeta_salida) / f"{base}_{idx:04d}.json"
        with out.open("w", encoding="utf-8") as f:
            json.dump(fila, f, indent=4, ensure_ascii=False)

    print(f"📁 JSON guardados en: {Path(carpeta_salida).resolve()}")

if __name__ == "__main__":
    # Ajusta estas rutas a tus ubicaciones reales:
    pdf_entrada = r"C:\Users\juans\Documents\pro\Model-Extract-information\document\resource\2020_2021\2020 2021_agenda_proyectos_ley_autor.pdf"
    carpeta_json = r"C:\Users\juans\Documents\pro\Model-Extract-information\document\2020_2021\proyectosAutor"

    procesar_pdf_texto_individual(pdf_entrada, carpeta_json)

CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, def

✅ Total de filas extraídas: 156
📁 JSON guardados en: C:\Users\juans\Documents\pro\Model-Extract-information\document\2020_2021\proyectosAutor
